# MSML 605 Final Project: Systems Benchmarking

Profiling LLM inference techniques on an A100 to understand the tradeoffs between throughput, memory, and power. The target model is Qwen 2.5 7B and the draft model (used for speculative decoding) is Qwen 2.5 0.5B.

I'm comparing four configurations: FP16 baseline (the reference point), speculative decoding with k=3 (the optimum from my 604 analysis), INT8 weight-only quantization through bitsandbytes, and INT4 (NF4) quantization. The roofline analysis at the end ties everything together by mapping achieved arithmetic intensity onto the A100 performance ceiling.

Note on quantization: the INT8 and INT4 cells failed at runtime due to a bitsandbytes/Triton compatibility issue on Colab that I couldn't resolve in time. The implementation is here for reference but didn't produce measurements. The final report frames quantization as roofline-predicted rather than measured, and discusses the failure honestly in section VI.D.


## Setup

Assumes A100 GPU runtime.

In [ ]:
import subprocess
print(subprocess.check_output(['nvidia-smi']).decode())

In [ ]:
# bitsandbytes is needed for INT8 and INT4 quantization
# Transformers 4.44.2 is what I've been using across all my experiments
!pip install -q transformers==4.44.2 accelerate bitsandbytes pynvml
print("dependencies installed")

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import time
import numpy as np
import json
import os
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
from pathlib import Path

# pynvml lets us read GPU power draw during inference
# the package is technically deprecated in favor of nvidia-ml-py but still works fine
try:
    import pynvml
    pynvml.nvmlInit()
    nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    HAS_NVML = True
except Exception as e:
    print(f"NVML not available: {e}")
    HAS_NVML = False

# fixing seeds so results are reproducible
torch.manual_seed(42)
np.random.seed(42)

device = "cuda"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DRAFT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
TARGET_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# Mount Drive so results persist across Colab sessions
# I lost data once when my runtime disconnected without this
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path("/content/drive/MyDrive/spec_decoding_results")
except:
    OUTPUT_DIR = Path("/content/spec_decoding_results")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
print(f"saving outputs to: {OUTPUT_DIR}")

## Measurement helpers

A few utility functions I use throughout. Nothing fancy but worth centralizing so timing is consistent across configurations.

The key thing about timing GPU ops in PyTorch is that you need `torch.cuda.synchronize()` before taking timestamps. CUDA operations are async by default, so without syncing, `time.perf_counter()` just measures when the op was submitted, not when it actually finished.

In [ ]:
def measure_model_footprint(model):
    # returns current GPU memory usage in GB
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    return torch.cuda.memory_allocated() / 1e9

def time_op(fn, warmup=3, trials=20):
    # time a function with proper GPU sync
    # warmup is important because first few calls include CUDA context init, JIT compilation, etc.
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(trials):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)  # ms
    return {'mean': np.mean(times), 'std': np.std(times),
            'min': np.min(times), 'max': np.max(times), 'all': times}

def get_gpu_power_watts():
    # reads instantaneous power from NVML
    # samples at call time, need multiple samples for a good average
    if not HAS_NVML: return None
    try:
        return pynvml.nvmlDeviceGetPowerUsage(nvml_handle) / 1000.0
    except:
        return None

@torch.no_grad()
def benchmark_generation(model, tokenizer, prompts, max_new_tokens=100, record_power=True):
    # runs autoregressive generation and measures throughput, memory, power
    # samples power during the decode loop (not prefill since prefill is a single big op)
    results = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

        power_samples = []
        t0 = time.perf_counter()

        # Prefill: one forward pass on the whole prompt to get the initial KV cache
        out = model(inputs.input_ids, use_cache=True)
        kv = out.past_key_values
        next_token = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)
        generated = [next_token.item()]

        # Decode loop: one token at a time, incrementally extending the KV cache
        for _ in range(max_new_tokens - 1):
            if record_power:
                p = get_gpu_power_watts()
                if p: power_samples.append(p)
            out = model(next_token, past_key_values=kv, use_cache=True)
            kv = out.past_key_values
            next_token = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)
            generated.append(next_token.item())
            if next_token.item() == tokenizer.eos_token_id:
                break

        torch.cuda.synchronize()
        total_time = time.perf_counter() - t0
        peak_mem = torch.cuda.max_memory_allocated() / 1e9
        avg_power = np.mean(power_samples) if power_samples else None

        results.append({
            'prompt': prompt,
            'total_time': total_time,
            'tokens': len(generated),
            'throughput': len(generated) / total_time,
            'peak_memory_gb': peak_mem,
            'avg_power_w': avg_power,
        })
    return results

print("helpers defined")

In [ ]:
# Five prompts covering different task types. Not a huge sample but enough
# for a systems-level comparison where we care about throughput/memory/power
# rather than domain-specific quality.
TEST_PROMPTS = [
    "Explain the key differences between supervised and unsupervised learning:",
    "Write a Python function to check if a number is prime:",
    "Summarize how neural networks learn in a paragraph:",
    "What are the main causes of climate change?",
    "Describe three ways to reduce food waste:",
]
print(f"{len(TEST_PROMPTS)} test prompts loaded")

## Configuration 1: FP16 baseline

Start with plain FP16 since it is the reference point everything else gets compared against. On A100 we have more than enough memory for the 7B model in FP16 (roughly 14 GB of weights) so this runs natively without any tricks.

In [ ]:
print("loading FP16 target model...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)
model_fp16 = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL,
    torch_dtype=torch.float16,
    device_map=device,
)
model_fp16.eval()
print(f"loaded in {time.time()-t0:.1f}s")
fp16_mem_static = measure_model_footprint(model_fp16)
print(f"static memory footprint: {fp16_mem_static:.2f} GB")

In [ ]:
# do one warmup pass so the first benchmarked run isn't slow due to CUDA init
print("warmup run...")
_ = benchmark_generation(model_fp16, tokenizer, [TEST_PROMPTS[0]], max_new_tokens=20, record_power=False)

print("running FP16 benchmarks...")
results_fp16 = benchmark_generation(model_fp16, tokenizer, TEST_PROMPTS, max_new_tokens=100)

print(f"\n{'Prompt':<50} | {'Throughput':>12} | {'Memory':>8} | {'Power':>7}")
print("-" * 90)
for r in results_fp16:
    prompt_short = r['prompt'][:47] + "..."
    if r['avg_power_w']:
        print(f"{prompt_short:<50} | {r['throughput']:>8.1f} t/s | {r['peak_memory_gb']:>6.2f} GB | {r['avg_power_w']:>5.1f} W")
    else:
        print(f"{prompt_short:<50} | {r['throughput']:>8.1f} t/s | {r['peak_memory_gb']:>6.2f} GB | {'N/A':>5}")

fp16_mean_throughput = np.mean([r['throughput'] for r in results_fp16])
fp16_mean_power = np.mean([r['avg_power_w'] for r in results_fp16 if r['avg_power_w']])
fp16_peak_mem = max(r['peak_memory_gb'] for r in results_fp16)
print(f"\nFP16 averages: {fp16_mean_throughput:.1f} tok/s, {fp16_mean_power:.1f} W, peak {fp16_peak_mem:.2f} GB")

## Configuration 2: Speculative decoding

Same speculative decoding implementation I built for the 604 project. The draft model proposes k=3 tokens each round (which was the optimum found by the optimization analysis), the target verifies them in one forward pass, and accepted tokens are committed.

One implementation detail worth calling out: the KV cache has to be truncated after each round to the accepted prefix length. This means constructing a new tuple of contiguous tensors each round, which is a real cost we measure below.

In [ ]:
# load the draft model alongside the target
# both fit on A100 easily (40GB or 80GB depending on which A100 you got)
print("loading draft model...")
t0 = time.time()
draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL, torch_dtype=torch.float16, device_map=device,
)
draft_model.eval()
print(f"loaded in {time.time()-t0:.1f}s")
print(f"total GPU memory now: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
@torch.no_grad()
def _truncate_kv(past_kv, target_len):
    # slices the KV cache back to target_len along the sequence dimension
    # need .contiguous() because PyTorch sometimes keeps views that don't play
    # well with subsequent model calls
    return tuple((k[:, :, :target_len, :].contiguous(), v[:, :, :target_len, :].contiguous())
                 for k, v in past_kv)

@torch.no_grad()
def speculative_decode(draft_model, target_model, tokenizer, prompt,
                       max_new_tokens=100, k=3, record_power=True):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_ids = inputs.input_ids
    full_ids = prompt_ids.clone()
    torch.cuda.reset_peak_memory_stats()

    torch.cuda.synchronize()
    t_start = time.perf_counter()
    power_samples = []

    # Prefill both models on the prompt. Each produces a KV cache we'll extend round by round.
    t_out = target_model(prompt_ids, use_cache=True)
    target_kv = t_out.past_key_values
    next_target_logits = t_out.logits[:, -1, :]
    d_out = draft_model(prompt_ids, use_cache=True)
    draft_kv = d_out.past_key_values
    next_draft_logits = d_out.logits[:, -1, :]

    n_generated = 0
    while n_generated < max_new_tokens:
        if record_power:
            p = get_gpu_power_watts()
            if p: power_samples.append(p)

        # draft proposes k tokens sequentially
        draft_tokens = []
        cur_logits = next_draft_logits
        cur_kv = draft_kv
        for _ in range(k):
            tok = torch.argmax(cur_logits, dim=-1, keepdim=True)
            draft_tokens.append(tok)
            d_out = draft_model(tok, past_key_values=cur_kv, use_cache=True)
            cur_kv = d_out.past_key_values
            cur_logits = d_out.logits[:, -1, :]

        # target verifies all k tokens in a single forward pass (this is the whole point of speculation)
        draft_seq = torch.cat(draft_tokens, dim=1)
        t_out = target_model(draft_seq, past_key_values=target_kv, use_cache=True)
        target_logits = t_out.logits
        target_kv_ext = t_out.past_key_values

        # Greedy acceptance: accept token i if target's argmax at that position matches draft's choice.
        # For stochastic sampling you'd do proper rejection sampling but greedy suffices here.
        n_accepted = 0
        next_token = None
        for i in range(k):
            tl = next_target_logits if i == 0 else target_logits[:, i-1, :]
            t_top = torch.argmax(tl, dim=-1)
            if t_top.item() == draft_tokens[i].item():
                n_accepted += 1
            else:
                # rejected at position i, use target's choice for this slot
                next_token = t_top.unsqueeze(0)
                break
        if n_accepted == k:
            # all k accepted, bonus token from target's distribution at position k
            next_token = torch.argmax(target_logits[:, k-1, :], dim=-1, keepdim=True)

        # commit the accepted tokens plus the next target token
        for i in range(n_accepted):
            full_ids = torch.cat([full_ids, draft_tokens[i]], dim=1)
        full_ids = torch.cat([full_ids, next_token], dim=1)
        n_generated += n_accepted + 1

        if next_token.item() == tokenizer.eos_token_id or n_generated >= max_new_tokens:
            break

        # truncate target KV to the accepted prefix length, then extend by one for the new token
        cur_target_len = target_kv_ext[0][0].shape[2]
        target_kv = _truncate_kv(target_kv_ext, cur_target_len - k + n_accepted)
        t_out = target_model(next_token, past_key_values=target_kv, use_cache=True)
        target_kv = t_out.past_key_values
        next_target_logits = t_out.logits[:, -1, :]

        # same for draft KV
        cur_draft_len = cur_kv[0][0].shape[2]
        draft_kv = _truncate_kv(cur_kv, cur_draft_len - k + n_accepted)
        d_out = draft_model(next_token, past_key_values=draft_kv, use_cache=True)
        draft_kv = d_out.past_key_values
        next_draft_logits = d_out.logits[:, -1, :]

    torch.cuda.synchronize()
    total_time = time.perf_counter() - t_start
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    avg_power = np.mean(power_samples) if power_samples else None

    return {
        'prompt': prompt,
        'total_time': total_time,
        'tokens': n_generated,
        'throughput': n_generated / total_time,
        'peak_memory_gb': peak_mem,
        'avg_power_w': avg_power,
    }

print("running speculative benchmarks (k=3, from 604 optimization analysis)...")
results_spec = []
for p in TEST_PROMPTS:
    r = speculative_decode(draft_model, model_fp16, tokenizer, p, max_new_tokens=100, k=3)
    results_spec.append(r)
    print(f"  {p[:40]}... | {r['throughput']:.1f} t/s | {r['peak_memory_gb']:.2f} GB | {r['avg_power_w']:.1f} W")

spec_mean_throughput = np.mean([r['throughput'] for r in results_spec])
spec_mean_power = np.mean([r['avg_power_w'] for r in results_spec if r['avg_power_w']])
spec_peak_mem = max(r['peak_memory_gb'] for r in results_spec)
print(f"\nSpeculative averages: {spec_mean_throughput:.1f} tok/s, {spec_mean_power:.1f} W, peak {spec_peak_mem:.2f} GB")

## Configuration 3: INT8 quantization

Now switching to INT8 weight-only quantization using bitsandbytes. The weights are stored as INT8 and dequantized on the fly during matrix multiplies. This gives roughly 2x memory savings compared to FP16.

I have to unload the FP16 model and draft model first to free up GPU memory. With INT8 we are reloading the same target model but with different precision.

In [ ]:
# free up memory from the FP16 experiments
print(f"GPU memory before cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
try:
    del model_fp16, draft_model
except:
    pass
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("loading INT8 quantized target...")
t0 = time.time()
bnb_config_int8 = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,  # default threshold
)
model_int8 = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL,
    quantization_config=bnb_config_int8,
    device_map={"": 0},   # explicitly assign all modules to GPU 0
    torch_dtype=torch.float16,
)
model_int8.eval()
print(f"loaded in {time.time()-t0:.1f}s")
int8_mem_static = torch.cuda.memory_allocated() / 1e9
print(f"INT8 static memory: {int8_mem_static:.2f} GB")
print(f"reduction vs FP16: {(1 - int8_mem_static/fp16_mem_static)*100:.1f}%")

In [ ]:
import json
from pathlib import Path
OUTPUT_DIR = Path("/content/drive/MyDrive/spec_decoding_results")

partial = {
    'fp16': {
        'throughput': fp16_mean_throughput,
        'memory_gb': fp16_peak_mem,
        'power_w': fp16_mean_power,
        'raw_results': results_fp16
    },
    'speculative': {
        'throughput': spec_mean_throughput,
        'memory_gb': spec_peak_mem,
        'power_w': spec_mean_power,
        'raw_results': results_spec
    },
}
with open(OUTPUT_DIR / 'systems_partial.json', 'w') as f:
    json.dump(partial, f, indent=2, default=str)
print("Saved")

In [ ]:
print("warmup run for INT8...")
_ = benchmark_generation(model_int8, tokenizer, [TEST_PROMPTS[0]], max_new_tokens=20, record_power=False)

print("running INT8 benchmarks...")
results_int8 = benchmark_generation(model_int8, tokenizer, TEST_PROMPTS, max_new_tokens=100)

for r in results_int8:
    print(f"  {r['prompt'][:40]}... | {r['throughput']:.1f} t/s | {r['peak_memory_gb']:.2f} GB | {r['avg_power_w']:.1f} W")

int8_mean_throughput = np.mean([r['throughput'] for r in results_int8])
int8_mean_power = np.mean([r['avg_power_w'] for r in results_int8 if r['avg_power_w']])
int8_peak_mem = max(r['peak_memory_gb'] for r in results_int8)
print(f"\nINT8 averages: {int8_mean_throughput:.1f} tok/s, {int8_mean_power:.1f} W, peak {int8_peak_mem:.2f} GB")

## Configuration 4: INT4 (NF4) quantization

Taking quantization further with NF4 (NormalFloat 4-bit). This was introduced in the QLoRA paper and represents weights more efficiently than plain INT4 by matching the normal distribution that pretrained weights tend to follow.

The tradeoff is pretty aggressive: each weight is stored in 4 bits, so we should see around 75% memory reduction compared to FP16. The interesting question is whether throughput holds up or gets worse due to the dequantization overhead.

In [ ]:
# free up INT8 before loading INT4
del model_int8
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("loading INT4 quantized target (NF4)...")
t0 = time.time()
# bnb_4bit_compute_dtype=float16 means the actual matmuls happen in FP16 after dequantization
# this is the standard QLoRA setup
bnb_config_int4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model_int4 = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL,
    quantization_config=bnb_config_int4,
    device_map=device,
)
model_int4.eval()
print(f"loaded in {time.time()-t0:.1f}s")
int4_mem_static = torch.cuda.memory_allocated() / 1e9
print(f"INT4 static memory: {int4_mem_static:.2f} GB")
print(f"reduction vs FP16: {(1 - int4_mem_static/fp16_mem_static)*100:.1f}%")

In [ ]:
print("warmup run for INT4...")
_ = benchmark_generation(model_int4, tokenizer, [TEST_PROMPTS[0]], max_new_tokens=20, record_power=False)

print("running INT4 benchmarks...")
results_int4 = benchmark_generation(model_int4, tokenizer, TEST_PROMPTS, max_new_tokens=100)

for r in results_int4:
    print(f"  {r['prompt'][:40]}... | {r['throughput']:.1f} t/s | {r['peak_memory_gb']:.2f} GB | {r['avg_power_w']:.1f} W")

int4_mean_throughput = np.mean([r['throughput'] for r in results_int4])
int4_mean_power = np.mean([r['avg_power_w'] for r in results_int4 if r['avg_power_w']])
int4_peak_mem = max(r['peak_memory_gb'] for r in results_int4)
print(f"\nINT4 averages: {int4_mean_throughput:.1f} tok/s, {int4_mean_power:.1f} W, peak {int4_peak_mem:.2f} GB")

## Comparison across all four configurations

Putting everything together. The interesting thing to watch is the relationship between throughput, memory, and power since each technique optimizes a different axis.

In [ ]:
# pull the numbers together into one comparison
print("Summary table")
print(f"{'Configuration':<20} | {'Throughput':>12} | {'Speedup':>8} | {'Memory':>10} | {'Power':>8} | {'Mem savings':>12}")
print("-" * 88)

configs = [
    ('FP16 Baseline', fp16_mean_throughput, fp16_peak_mem, fp16_mean_power),
    ('Speculative k=3', spec_mean_throughput, spec_peak_mem, spec_mean_power),
    ('INT8', int8_mean_throughput, int8_peak_mem, int8_mean_power),
    ('INT4 (NF4)', int4_mean_throughput, int4_peak_mem, int4_mean_power),
]

for name, tps, mem, pwr in configs:
    speedup = tps / fp16_mean_throughput
    mem_reduction = (1 - mem/fp16_peak_mem) * 100
    print(f"{name:<20} | {tps:>10.1f} t/s | {speedup:>6.2f}x | {mem:>7.2f} GB | {pwr:>6.1f} W | {mem_reduction:>+10.1f}%")

# save the raw data for the report
systems_results = {
    'fp16': {'throughput': fp16_mean_throughput, 'memory_gb': fp16_peak_mem,
             'power_w': fp16_mean_power, 'raw_results': results_fp16},
    'speculative': {'throughput': spec_mean_throughput, 'memory_gb': spec_peak_mem,
                    'power_w': spec_mean_power, 'raw_results': results_spec},
    'int8': {'throughput': int8_mean_throughput, 'memory_gb': int8_peak_mem,
             'power_w': int8_mean_power, 'raw_results': results_int8},
    'int4': {'throughput': int4_mean_throughput, 'memory_gb': int4_peak_mem,
             'power_w': int4_mean_power, 'raw_results': results_int4},
}

with open(OUTPUT_DIR / 'systems_comparison.json', 'w') as f:
    json.dump(systems_results, f, indent=2, default=str)
print(f"\nsaved to {OUTPUT_DIR / 'systems_comparison.json'}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# four-panel comparison: throughput, memory, power, energy per token
# energy per token is power / throughput, gives joules per token
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

names = [c[0] for c in configs]
throughputs = [c[1] for c in configs]
memories = [c[2] for c in configs]
powers = [c[3] for c in configs]
energy_per_token = [p / t for t, p in zip(throughputs, powers)]

colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

axes[0, 0].bar(names, throughputs, color=colors)
axes[0, 0].set_ylabel('Throughput (tokens/sec)')
axes[0, 0].set_title('Throughput')
axes[0, 0].axhline(y=fp16_mean_throughput, color='gray', linestyle='--', alpha=0.5, label='FP16 baseline')
axes[0, 0].legend()
for i, v in enumerate(throughputs):
    axes[0, 0].text(i, v + 0.5, f'{v:.1f}', ha='center')

axes[0, 1].bar(names, memories, color=colors)
axes[0, 1].set_ylabel('Peak GPU memory (GB)')
axes[0, 1].set_title('Memory footprint')
for i, v in enumerate(memories):
    axes[0, 1].text(i, v + 0.3, f'{v:.2f}', ha='center')

axes[1, 0].bar(names, powers, color=colors)
axes[1, 0].set_ylabel('Average power (W)')
axes[1, 0].set_title('Power draw during inference')
for i, v in enumerate(powers):
    axes[1, 0].text(i, v + 3, f'{v:.0f}', ha='center')

axes[1, 1].bar(names, energy_per_token, color=colors)
axes[1, 1].set_ylabel('Energy per token (J/token)')
axes[1, 1].set_title('Energy efficiency (lower is better)')
for i, v in enumerate(energy_per_token):
    axes[1, 1].text(i, v + 0.2, f'{v:.2f}', ha='center')

for ax in axes.flat:
    plt.setp(ax.get_xticklabels(), rotation=15, ha='right')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'systems_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"saved plot to {OUTPUT_DIR / 'systems_comparison.png'}")

## Roofline analysis

The roofline model is a really clean way to visualize whether a workload is limited by compute or memory bandwidth. It plots achieved performance (in FLOPS) against arithmetic intensity (FLOPs per byte of memory transferred).

For the A100 specifically:
- Peak FP16 tensor core throughput is around 312 TFLOPS
- Peak memory bandwidth is 2 TB/s for HBM2e
- The ridge point where you transition from memory-bound to compute-bound is at 312 / 2, or around 156 FLOPs/byte

At batch size 1, LLM inference has very low arithmetic intensity because you read all the weights once per forward pass but only do O(num_params) FLOPs. This puts us deep in the memory-bound region, which turns out to be the main reason speculative decoding does not help much on modern GPUs at batch 1.

In [ ]:
# A100 specs (from NVIDIA datasheet)
A100_PEAK_FLOPS_FP16 = 312e12   # 312 TFLOPS for FP16 tensor cores
A100_PEAK_BANDWIDTH = 2.0e12    # 2 TB/s HBM2e

NUM_PARAMS = 7.62e9  # Qwen 2.5 7B actual parameter count

# rough FLOPs estimate: 2*N for a forward pass (multiply-accumulate pairs)
# memory access: read all weights once (N * bytes_per_param)
configs_roofline = {
    'FP16 Baseline': {
        'bytes_per_param': 2,
        'flops_per_token': 2 * NUM_PARAMS,
        'throughput': fp16_mean_throughput
    },
    'Speculative k=3': {
        # speculative effectively processes ~4 tokens per target forward pass
        # but we need to count the draft work too (3 * draft_flops is small)
        'bytes_per_param': 2,
        'flops_per_token': 2 * NUM_PARAMS * (1 + 4/3.83),
        'throughput': spec_mean_throughput
    },
    'INT8': {
        'bytes_per_param': 1,
        'flops_per_token': 2 * NUM_PARAMS,
        'throughput': int8_mean_throughput
    },
    'INT4 (NF4)': {
        'bytes_per_param': 0.5,
        'flops_per_token': 2 * NUM_PARAMS,
        'throughput': int4_mean_throughput
    },
}

for name, c in configs_roofline.items():
    bytes_per_token = NUM_PARAMS * c['bytes_per_param']
    c['AI'] = c['flops_per_token'] / bytes_per_token
    c['achieved_flops'] = c['flops_per_token'] * c['throughput']
    c['compute_util'] = c['achieved_flops'] / A100_PEAK_FLOPS_FP16 * 100
    print(f"{name:<20}: AI = {c['AI']:.2f} FLOPs/byte, "
          f"achieved {c['achieved_flops']/1e12:.1f} TFLOPS ({c['compute_util']:.2f}% of peak)")

In [ ]:
# generate the roofline plot
fig, ax = plt.subplots(figsize=(12, 8))

AI_range = np.logspace(-1, 3, 1000)
memory_roof = A100_PEAK_BANDWIDTH * AI_range
compute_roof = np.full_like(AI_range, A100_PEAK_FLOPS_FP16)
roof = np.minimum(memory_roof, compute_roof)

ax.loglog(AI_range, roof, 'k-', linewidth=2, label='Roofline')
ax.loglog(AI_range, memory_roof, 'k--', linewidth=1, alpha=0.5, label='Memory bound: 2 TB/s')
ax.loglog(AI_range, compute_roof, 'k:', linewidth=1, alpha=0.5, label='Compute bound: 312 TFLOPS')

# plot our four configurations on the roofline
markers = ['o', 's', '^', 'D']
for i, (name, c) in enumerate(configs_roofline.items()):
    ax.loglog(c['AI'], c['achieved_flops'], marker=markers[i], markersize=15,
              color=colors[i], label=name, markeredgecolor='black', markeredgewidth=1)

ridge_point = A100_PEAK_FLOPS_FP16 / A100_PEAK_BANDWIDTH
ax.axvline(x=ridge_point, color='gray', linestyle=':', alpha=0.3)
ax.text(ridge_point * 1.2, 1e10, f'Ridge: {ridge_point:.0f}\nFLOPs/byte', fontsize=9)

ax.set_xlabel('Arithmetic intensity (FLOPs/byte)', fontsize=12)
ax.set_ylabel('Performance (FLOPS)', fontsize=12)
ax.set_title('Roofline: Qwen 2.5 7B on A100 (batch=1)', fontsize=13)
ax.legend(loc='lower right')
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(0.1, 1000)
ax.set_ylim(1e9, 1e15)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roofline.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"saved to {OUTPUT_DIR / 'roofline.png'}")

# interpretation
print("\nTakeaway: all four configurations land way below the compute roof.")
print("Compute utilization ranges from "
      f"{min(c['compute_util'] for c in configs_roofline.values()):.2f}% to "
      f"{max(c['compute_util'] for c in configs_roofline.values()):.2f}% of peak A100 FP16.")
print("\nThis makes sense. At batch 1 we are deeply memory-bound, so the A100")
print("has tons of compute headroom but the HBM bandwidth is what we are actually using.")
print("Explains why speculative decoding doesn't provide real wall-clock speedup")
print("in this regime: speculation helps when the target is compute-bound,")
print("but here it is the memory bandwidth that limits per-token inference.")

## PyTorch profiler: where does the time actually go?

The profiler gives kernel-level timing so I can see exactly which CUDA operations dominate during decoding. For the FP16 baseline I would expect the matrix multiplies (attention Q/K/V projections, MLP gate/up/down) to dominate, but the size of each contribution matters for understanding the bottleneck.

Need to reload the FP16 model since it was unloaded earlier to make room for quantized versions.

In [ ]:
print("reloading FP16 for profiling...")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL, torch_dtype=torch.float16, device_map="cuda",
)
model_fp16.eval()
print("ready")

In [ ]:
torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("reloading FP16 for profiling...")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL, torch_dtype=torch.float16, device_map="cuda",
)
model_fp16.eval()
print("ready")

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

prompt = "Explain machine learning briefly:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# do prefill outside the profiler so we're only measuring decode steps
with torch.no_grad():
    out = model_fp16(inputs.input_ids, use_cache=True)
    past_kv = out.past_key_values
    next_token = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)

print("profiling 10 decode steps...")
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
             record_shapes=True) as prof:
    with torch.no_grad():
        for step in range(10):
            with record_function(f"decode_step_{step}"):
                out = model_fp16(next_token, past_key_values=past_kv, use_cache=True)
                past_kv = out.past_key_values
                next_token = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)

# show top 15 kernels by total CUDA time
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))

In [ ]:
# extract kernel timings for plotting and the report
key_avgs = prof.key_averages()
kernel_data = []
for k in key_avgs:
    if k.self_device_time_total > 0:
        kernel_data.append({
            'name': k.key,
            'cuda_time_us': k.self_device_time_total,
            'count': k.count,
            'avg_us': k.self_device_time_total / k.count if k.count > 0 else 0,
        })

kernel_data.sort(key=lambda x: x['cuda_time_us'], reverse=True)
total_cuda_time = sum(k['cuda_time_us'] for k in kernel_data)

print(f"\nTotal CUDA time: {total_cuda_time/1000:.2f} ms across 10 decode steps\n")
print(f"{'Rank':>4} | {'Kernel':<50} | {'Total (ms)':>12} | {'% Total':>8} | {'Count':>6} | {'Avg (us)':>10}")
print("-" * 110)
for i, k in enumerate(kernel_data[:15]):
    pct = k['cuda_time_us'] / total_cuda_time * 100
    print(f"{i+1:>4} | {k['name'][:48]:<50} | {k['cuda_time_us']/1000:>10.2f} | {pct:>7.1f}% | {k['count']:>6} | {k['avg_us']:>8.1f}")

# save so we can reference in the report
profile_data = {
    'total_cuda_time_ms': total_cuda_time / 1000,
    'n_decode_steps': 10,
    'top_kernels': kernel_data[:20],
}
with open(OUTPUT_DIR / 'profile_fp16.json', 'w') as f:
    json.dump(profile_data, f, indent=2)
print(f"\nsaved to {OUTPUT_DIR / 'profile_fp16.json'}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

In [ ]:
# plot top kernels as a horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 6))

top_n = 10
top_kernels = kernel_data[:top_n]
others_time = sum(k['cuda_time_us'] for k in kernel_data[top_n:])

# truncate long kernel names for readability
names_display = [k['name'][:40] + ('...' if len(k['name']) > 40 else '') for k in top_kernels]
names_display.append('Other')
times = [k['cuda_time_us']/1000 for k in top_kernels]
times.append(others_time/1000)

ax.barh(names_display[::-1], times[::-1], color=plt.cm.viridis(np.linspace(0.2, 0.9, len(times))))
ax.set_xlabel('CUDA time (ms)')
ax.set_title('Top CUDA kernels: FP16 baseline over 10 decode steps')
ax.grid(axis='x', alpha=0.3)

for i, v in enumerate(times[::-1]):
    ax.text(v + 0.1, i, f'{v:.1f} ms', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'kernel_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"saved to {OUTPUT_DIR / 'kernel_breakdown.png'}")

## Latency distribution

Average throughput hides a lot. For real-world deployment, tail latency (p99 especially) is often what users actually feel. A model with 10 tok/s average but a long tail is worse than a model with 8 tok/s average and consistent latency.

Running 200 decode steps to get a proper distribution.

In [ ]:
def measure_per_step_latencies(model, tokenizer, prompt, n_steps=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(inputs.input_ids, use_cache=True)
        past_kv = out.past_key_values
        next_token = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)

    step_times = []
    torch.cuda.synchronize()
    for _ in range(n_steps):
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(next_token, past_key_values=past_kv, use_cache=True)
            past_kv = out.past_key_values
            next_token = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)
        torch.cuda.synchronize()
        step_times.append((time.perf_counter() - t0) * 1000)
    return step_times

print("measuring 200 decode step latencies for FP16...")
latencies_fp16 = measure_per_step_latencies(model_fp16, tokenizer,
                                             "Explain photosynthesis:", n_steps=200)

lat_arr = np.array(latencies_fp16)
print(f"\nPer-step latency statistics (ms):")
print(f"  Mean:   {lat_arr.mean():.2f}")
print(f"  Median: {np.median(lat_arr):.2f}")
print(f"  p50:    {np.percentile(lat_arr, 50):.2f}")
print(f"  p90:    {np.percentile(lat_arr, 90):.2f}")
print(f"  p99:    {np.percentile(lat_arr, 99):.2f}")
print(f"  Max:    {lat_arr.max():.2f}")
print(f"  Std:    {lat_arr.std():.2f}")

# side-by-side histogram and time series
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(lat_arr, bins=30, color='#2E86AB', edgecolor='black', alpha=0.7)
axes[0].axvline(np.median(lat_arr), color='red', linestyle='--',
                label=f'Median: {np.median(lat_arr):.1f} ms')
axes[0].axvline(np.percentile(lat_arr, 99), color='orange', linestyle='--',
                label=f'p99: {np.percentile(lat_arr, 99):.1f} ms')
axes[0].set_xlabel('Per-step latency (ms)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('FP16 baseline: per-step latency distribution')
axes[0].legend()

axes[1].plot(lat_arr, alpha=0.5)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Per-step latency over 200 consecutive decodes')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'latency_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"saved to {OUTPUT_DIR / 'latency_distribution.png'}")

In [ ]:
# Save profiler and latency data alongside the partial systems data
import json
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/spec_decoding_results")

# Profiler data should already be saved from the cell, verify
print("Files in output dir:")
for f in OUTPUT_DIR.iterdir():
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
from google.colab import files
import os
for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.png'):
        files.download(str(OUTPUT_DIR / f))

## Wrap-up

That's the systems data for my 605 report. A few things stood out as I went through the analysis.

The roofline prediction held up. FP16 baseline at A100 sits deep in the memory-bound regime (~2 FLOPs/byte vs the 156 FLOPs/byte ridge point), so the upper bound on throughput is roughly the HBM bandwidth divided by bytes per token. The empirical 26.4 tok/s lines up with that ceiling.

Speculative decoding underperformed FP16 even at the optimal k. This is the most interesting finding because it goes against the literature's framing of speculative decoding as a default acceleration technique. The reason traces back to the per-token time ratio between the 0.5B draft and the 7B target. On A100 it's only 1.22x, not the 14x you'd expect from parameter count, because both models pay the same fixed per-token overhead from kernel launches, attention, and KV-cache reads. The textbook speedup formula plugs that 1.22x ratio in and predicts a 0.55x slowdown, which matches my measured 0.54x almost exactly.

The PyTorch profiler confirmed it. The aten::mm kernel dominates at 111.8 ms across 10 decode steps, GPU utilization stays under 5%, and the latency distribution is tight (median 47.8 ms, p99 52.0 ms). Three independent signals all point to memory-bound operation.

The quantization gap is what didn't work. INT8 and INT4 implementations are here but blocked by the bitsandbytes/Triton issue. The roofline analysis still tells us what they should give: shifting bytes per parameter from 2 to 1 (INT8) or 0.5 (INT4) doubles or quadruples the effective bandwidth, which the report covers as a prediction.
